# I8. Polarisation

The eighth notebook of the introductory track ({doc}`../index`). It writes
`poldx`, A1 small's antennas and schedule with 256 channels and an
instrument with leakage on every antenna (157 MB), calibrates it in full
polarisation with CARACal's strategy, and measures the polarisation
calibrator's fraction and angle against the truth. About a minute.

**Learning goals.** After this notebook you can

- write the four correlations of linear feeds in terms of Stokes I, Q, U, V;
- compute the polarisation fraction and angle, and say what Faraday rotation
  does to the angle;
- name the polarisation calibration terms: leakage Df, cross-hand delay
  KCROSS, cross-hand phase Xf, and why the parallactic angle separates the
  source's polarisation from the instrument's;
- check calibrated polarisation against a truth, and make Stokes images.

In [ ]:
%matplotlib inline
import json
import os
import shlex
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits

from rang import cal_pol, core_ms, core_ms_write
from rang import simtruth as st
from rang.notebook import data

os.environ.setdefault("RAYON_NUM_THREADS", "2")
WORK = data.default_cache_dir() / "intro-i8"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
COMMANDS = {"rang-core": [sys.executable, "-m", "rang.core_cli"], "gemclean": [sys.executable, "-m", "rang.gemclean"], "python": [sys.executable]}


def sh(line, tail=1):
    print("$", line)
    argv = shlex.split(line)
    p = subprocess.run(COMMANDS.get(argv[0], [argv[0]]) + argv[1:], cwd=WORK, capture_output=True, text=True, encoding="utf-8")
    lines = (p.stdout + p.stderr).rstrip().splitlines()
    print("\n".join(lines[-tail:] if tail else lines))
    assert p.returncode == 0, f"exit {p.returncode}"


sh("python -m rang.simtruth_ms generate poldx . --workers 2")
MS = WORK / "a1-poldx.ms"
d = st.design("poldx")
z = np.load(WORK / "truth.npz")
freq = z["freq_hz"]
print(f"{MS.name}: {sum(p.stat().st_size for p in MS.rglob('*') if p.is_file()) / 1e6:.0f} MB, {d.nchan} channels")

## 1. Stokes parameters and linear feeds

Partially polarised radiation is described by four Stokes parameters: I
(total intensity), Q and U (linear polarisation) and V (circular). The
linear polarisation has fraction p = √(Q² + U²) / I and angle
χ = ½ atan2(U, Q). An X and a Y dipole correlated four ways give (I4)

XX = I + Q,  XY = U + iV,  YX = U − iV,  YY = I − Q

in the frame of the sky. Magnetised plasma along the line of sight rotates
the angle with wavelength, χ(λ) = χ₀ + RM λ² (**Faraday rotation**; RM is
the rotation measure).

A1's polarisation calibrator 3C286-SIM follows 3C 286, the standard angle
calibrator:

In [ ]:
x = st.XCAL
stokes = st.calibrator_stokes(d)[1]  # (channel, I Q U V)
p_true = np.hypot(stokes[:, 1], stokes[:, 2]) / stokes[:, 0]
chi_true = np.degrees(0.5 * np.arctan2(stokes[:, 2], stokes[:, 1]))
print(f"I = {x['fluxdensity']} Jy at {x['reffreq_hz'] / 1e6:.0f} MHz, p = {x['p']}, chi = {x['chi_deg']} deg, RM = {x['rm']} rad/m^2")
print(f"from the Stokes spectrum: p {p_true.min():.4f}-{p_true.max():.4f}, chi {chi_true.min():.2f}-{chi_true.max():.2f} deg across the band")
rm = 50.0  # an example
lam2 = (st.C / freq) ** 2
print(f"for comparison, RM = {rm:.0f} rad/m^2 would rotate chi by {np.degrees(rm * (lam2[0] - lam2[-1])):.0f} deg from 856 to 1712 MHz")

## 2. The parallactic angle

MeerKAT's dishes are on alt-azimuth mounts: the feed turns relative to the
sky as a source is tracked, by the **parallactic angle** q. The instrument's
leakage and cross-hand phase stay fixed to the feed; the source's
polarisation turns with q. That difference is what lets a solver tell them
apart: observe the polarisation calibrator over a wide range of q.

In [ ]:
times, scans, fields, _ = st.schedule(d)
xyz, _, _ = st.antenna_table()
name, ra, dec = st.FIELDS[1][:3]
sel = fields == 1
q = cal_pol.parallactic_angle(times[sel], np.radians(ra), np.radians(dec), xyz[d.antennas[0]])
print(f"{name}: {len(set(scans[sel]))} scans, parallactic angle at m000 from {np.degrees(q).min():.1f} to {np.degrees(q).max():.1f} deg (CASA frame)")
print(f"matches the truth's: {np.allclose(q, z['parallactic_angle'][sel, 0])}")

## 3. The polarisation terms in poldx

On top of K, G and B (I4, I6), each antenna's receptors leak into each other
(d_x, d_y, per channel), and the array has one cross-hand delay and phase
between the X and Y signal chains. In CASA's gauge for m000:

In [ ]:
leak = z["gauge0_leakage"]
pd.DataFrame(
    [
        {"term": "Df, |d|", "truth": f"median {np.median(np.abs(leak)):.4f}, max {np.abs(leak).max():.4f}"},
        {"term": "Xf phase", "truth": f"{np.degrees(np.angle(z['gauge0_xf'])).min():.1f} to {np.degrees(np.angle(z['gauge0_xf'])).max():.1f} deg"},
        {"term": "KCROSS (array X-Y delay)", "truth": f"{d.tau_c_ns} ns raw"},
        {"term": "d_x of the refant m000", "truth": f"{np.abs(z['raw_leakage'][0, :, 0]).max():.4f} (not zero: what poldx tests)"},
    ]
)

## 4. Calibrate

CARACal's polarisation strategy 1, as task C6w wired it into
`rang-core calibrate` ({gh}`research/c6w-polwire.md`): after K, G, B on the
flux calibrator,

- **Gpol**, phase-only gains on 3C286-SIM against its model (CASA's
  `nrao` 3C 286), with P applied;
- **KCROSS**, the X−Y delay, on 3C286-SIM;
- **Xf**, the X−Y phase per channel, on 3C286-SIM: needs the polarised
  source seen at several parallactic angles;
- **Df**, the leakages, on the unpolarised flux calibrator, where any
  cross-hand signal is instrumental.

`apply.parang: true` rotates the corrected data back to the sky frame. To
keep this notebook about polarisation, the RFI is flagged with A1's truth
mask (an oracle, I5) instead of a flagger, as C6w did:

In [ ]:
mask = np.unpackbits(np.load(WORK / "a1-poldx.rfi_mask.npz")["any"], axis=1, count=d.nchan).astype(bool)
with core_ms_write.open_table(str(MS), readonly=False) as t:
    t.putcol("FLAG", t.getcol("FLAG") | mask[..., None])
config = """\
schema: rang-config
version: 1
fields: {fcal: [J1939-6342], bpcal: [J1939-6342], gcal: [SIM-GCAL], xcal: [3C286-SIM], target: [SIM-TARGET]}
calibrate:
  model: {source: setjy, standard: Stevens-Reynolds 2016}
  refant: m000
  uvrange: ">150"
  primary:
    field: [fcal]
    terms:
      - {type: K, name: K, solint: inf}
      - {type: G, name: G, solint: inf, calmode: ap, preapply: [K], interp: [linear]}
      - {type: B, name: B, solint: inf, combine: scan, fillgaps: 70, preapply: [K, G], interp: [linear, linear]}
  polcal:
    field: [xcal]
    model: {source: nrao, name: 3C286}
    feed_angle_rotation: -90
    terms:
      - {type: Gpol, name: Gpol, solint: 60s, calmode: p, parang: true, preapply: [K, G, B], interp: [linear, linear, linear]}
      - {type: KCROSS, name: Kcross, solint: inf, parang: true, preapply: [K, G, B, Gpol], interp: [linear, linear, linear, linear]}
      - {type: Xf, name: Xf, solint: inf, combine: scan, parang: true, preapply: [K, G, B, Gpol, Kcross],
         interp: [linear, linear, linear, linear, nearest]}
      - {type: Df, name: Df, field: [fcal], solint: inf, combine: scan, parang: true, clip: [-0.6, 0.6],
         preapply: [K, G, B, Kcross, Xf], interp: [linear, linear, linear, nearest, nearest]}
apply:
  calwt: false
  parang: true
  targets:
    - field: [xcal]
      tables: [K, G, B, Gpol, Kcross, Xf, Df]
      interp: [linear, linear, linear, linear, nearest, nearest, nearest]
"""
(WORK / "config.yaml").write_text(config, encoding="utf-8")
sh("rang-core calibrate config.yaml a1-poldx.ms")
sh("rang-core apply config.yaml a1-poldx.ms")
CAL = WORK / "a1-poldx.ms.rang-run" / "calibrate"

## 5. The solved terms against the truth

In [ ]:
with core_ms.open_table(str(CAL / "Df")) as t:
    d_sol, d_ant, d_flag = t.getcol("CPARAM"), t.getcol("ANTENNA1"), t.getcol("FLAG")
with core_ms.open_table(str(CAL / "Xf")) as t:
    x_sol, x_flag = t.getcol("CPARAM")[0, :, 0], t.getcol("FLAG")[0, :, 0]
with core_ms.open_table(str(CAL / "Kcross")) as t:
    kc_scan = t.getcol("FPARAM")[:, 0, 0]  # ns on X, one row per antenna and 3C286-SIM scan
nu0 = float(z["nu0_hz"])
d_err = np.ma.masked_array(np.abs(d_sol - leak[d_ant]), d_flag)
print(f"Df: |d_solved - d_true| median {np.ma.median(d_err):.4f}, 95th percentile {np.percentile(d_err.compressed(), 95):.4f} (true |d| median {np.median(np.abs(leak)):.4f}); flagged {d_flag.mean():.1%}")
# KCROSS and Xf share the X-Y phase: compare their product with the truth's.
true_tau_c = np.polyfit(freq - nu0, np.unwrap(np.angle(z["gauge0_kcross"])), 1)[0] / (2 * np.pi) * 1e9
print(f"KCROSS per scan: {np.round(np.unique(kc_scan), 3)} ns; the truth's effective X-Y delay {true_tau_c:.3f} ns")
kc = np.exp(2j * np.pi * np.median(kc_scan) * 1e-9 * (freq - nu0))
xy_err = np.degrees(np.angle(kc * x_sol / (z["gauge0_kcross"] * z["gauge0_xf"])))
xy_err = np.ma.masked_array(xy_err, x_flag)
print(f"KCROSS·Xf phase minus the truth's: median {np.ma.median(xy_err):+.2f} deg, median |.| {np.ma.median(np.abs(xy_err)):.2f} deg, max |.| {np.abs(xy_err).max():.2f} deg")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
a = 5
for h, hand in enumerate(("d_x", "d_y")):
    axes[0].plot(freq / 1e6, np.ma.masked_array(d_sol[a, :, h].real, d_flag[a, :, h]), ".", ms=3, label=f"Re {hand}, solved")
    axes[0].plot(freq / 1e6, leak[d_ant[a], :, h].real, "-", lw=1, label=f"Re {hand}, truth")
axes[0].set(xlabel="frequency (MHz)", ylabel="leakage", title=f"Df of m{d.antennas[d_ant[a]]:03d}")
axes[0].legend(fontsize=7)
axes[1].plot(freq / 1e6, np.degrees(np.ma.masked_array(np.angle(kc * x_sol), x_flag)), ".", ms=3, label="solved")
axes[1].plot(freq / 1e6, np.degrees(np.angle(z["gauge0_kcross"] * z["gauge0_xf"])), "-", lw=1, label="truth")
axes[1].set(xlabel="frequency (MHz)", ylabel="phase (deg)", title="X−Y phase, KCROSS·Xf")
axes[1].legend()
plt.show()

## 6. The calibrator's polarisation

With everything applied, 3C286-SIM's corrected correlations give its Stokes
parameters directly. Per channel, the mean over unflagged cells, then
I = Re(XX + YY)/2, Q = Re(XX − YY)/2, U = Re(XY + YX)/2 (C6w's statistic):

In [ ]:
with core_ms.open_table(str(MS)) as t:
    field = t.getcol("FIELD_ID")
    rows = np.flatnonzero(field == 1)
    span = (int(rows[0]), int(rows[-1] - rows[0] + 1))
    v = t.getcol("CORRECTED_DATA", *span)
    f = t.getcol("FLAG", *span)
keep = field[span[0] : span[0] + span[1]] == 1
m = np.ma.masked_array(v[keep], f[keep]).mean(axis=0)  # (channel, correlation)
I = (m[:, 0] + m[:, 3]).real / 2
Q = (m[:, 0] - m[:, 3]).real / 2
U = (m[:, 1] + m[:, 2]).real / 2
p = np.ma.sqrt(Q**2 + U**2) / I
chi = np.degrees(0.5 * np.ma.arctan2(U, Q))
print(f"p: median {np.ma.median(p):.4f} (truth {x['p']}); chi: median {np.ma.median(chi):.2f} deg (truth {x['chi_deg']} deg)")
print(f"|chi - truth| per channel: median {np.ma.median(np.abs(chi - x['chi_deg'])):.2f} deg, 95th percentile {np.percentile(np.abs(chi - x['chi_deg']).compressed(), 95):.2f} deg")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(freq / 1e6, p, ".", ms=3)
axes[0].axhline(x["p"], color="k", lw=1)
axes[0].set(xlabel="frequency (MHz)", ylabel="p", title="polarisation fraction")
axes[1].plot(freq / 1e6, chi, ".", ms=3)
axes[1].axhline(x["chi_deg"], color="k", lw=1)
axes[1].set(xlabel="frequency (MHz)", ylabel="chi (deg)", title="polarisation angle")
plt.show()

C6w froze its tolerances before measuring: per-channel |χ − 33°| median at
most 1.11° and |median p − 0.095| at most 0.0015
({gh}`research/c6w-polwire.md`, gate 4). This run differs from C6w's (the
frame is the default `casa`, nothing else is solved), so the comparison is
a check, not a re-run of the gate.

## 7. Stokes images

`gemclean -pol IQUV` images each Stokes parameter; `-join-polarizations`
finds peaks on their combined power, so Q and U are cleaned where I is.
GeMClean writes a model back into MODEL_DATA for Stokes I only and refuses
by name otherwise, hence `-no-update-model-required`:

In [ ]:
sh("gemclean -name pol -size 256 256 -scale 3asec -weight briggs 0 -niter 2000 -auto-threshold 3 -pol IQUV -join-polarizations -no-update-model-required -field 1 -j 2 a1-poldx.ms")
maps = {}
for s_name in "IQUV":
    with fits.open(WORK / f"pol-{s_name}-image.fits") as h:
        maps[s_name] = np.squeeze(h[0].data)
c = 128
peak = {k: float(v[c, c]) for k, v in maps.items()}
print({k: round(v, 4) for k, v in peak.items()})
print(f"image: p = {np.hypot(peak['Q'], peak['U']) / peak['I']:.4f}, chi = {np.degrees(0.5 * np.arctan2(peak['U'], peak['Q'])):.2f} deg")
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (k, im) in zip(axes, maps.items()):
    lim = np.abs(im[c - 32 : c + 32, c - 32 : c + 32]).max()
    ax.imshow(im[c - 32 : c + 32, c - 32 : c + 32], origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(f"Stokes {k}")
plt.show()

## Exercises

1. Remove Df from the applied tables. How much Stokes V and how much
   spurious Q, U does the unpolarised flux calibrator show? (Apply to it
   too.)
2. Image only the first two 3C286-SIM scans. Does Xf still come out right
   when the parallactic angle range is small? (`--scans` of the generator.)
3. Set `feed_angle_rotation: 0`. What happens to χ, and why is the answer
   a rotation and not noise?
4. Give 3C286-SIM a rotation measure of 30 rad/m² (`st.XCAL`) and regenerate.
   Fit χ(λ²) to recover it.

## Where next

- {doc}`i9-time-domain`: pulsars and transients.
- {gh}`research/c6-polterms.md` and {gh}`research/c6w-polwire.md`: the
  polarisation chain and its checks against CASA;
  {gh}`research/g9-stokes.md`: GeMClean's Stokes imaging.
- {doc}`i4-measurement-equation` for the Jones terms;
  {doc}`00-ms-to-image` section 6 on the `polcal` section.